In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Běžně používané parametry pro transpilaci

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Tato stránka popisuje některé z běžněji používaných parametrů pro lokální transpilaci. Tyto parametry se konfigurují pomocí argumentů funkce [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) nebo [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Stupeň aproximace
Pomocí stupně aproximace můžeš určit, jak přesně má výsledný Circuit odpovídat požadovanému (vstupnímu) Circuit. Jedná se o desetinné číslo v rozsahu (0,0–1,0), kde 0,0 znamená maximální aproximaci a 1,0 (výchozí hodnota) znamená žádnou aproximaci. Nižší hodnoty vyměňují přesnost výstupu za snazší provedení (tedy méně Gate). Výchozí hodnota je 1,0.

Při syntéze dvoukubitových unitárních operací (používané v počátečních fázích všech úrovní a ve fázi optimalizace s optimalizační úrovní 3) tato hodnota určuje cílovou věrnost výstupní dekompozice. Tedy jak velká chyba je zavedena při převodu maticové reprezentace Circuit na diskrétní Gate. Pokud je stupeň aproximace nižší (více aproximace), výstupní Circuit ze syntézy se bude více lišit od vstupní matice, ale pravděpodobně bude mít méně Gate (protože libovolnou dvoukubitovou operaci lze dokonale rozložit pomocí nejvýše tří CX Gate) a bude snazší ho spustit.

Když je stupeň aproximace menší než 1,0, mohou být syntetizovány Circuit s jedním nebo dvěma CX Gate, což vede k menší chybě z hardwaru, ale větší z aproximace. Protože CX je nejnákladnější Gate z hlediska chyb, může být výhodné snížit jejich počet za cenu věrnosti při syntéze (tato technika byla použita ke zvýšení kvantového objemu na zařízeních IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Jako příklad vygenerujeme náhodnou dvoukubitovou `UnitaryGate`, která bude syntetizována v počáteční fázi. Nastavením `approximation_degree` na hodnotu nižší než 1,0 může být vygenerován přibližný Circuit. Musíme také zadat `basis_gates`, aby metoda syntézy věděla, které Gate může pro přibližnou syntézu použít.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

Výsledkem je `2`, protože aproximace vyžaduje méně CX Gate.

<span id="seed"></span>
## Seed generátoru náhodných čísel
Některé části Transpileru jsou stochastické, takže opakované transpilace mohou vracet různé výsledky. Chceš-li získat reprodukovatelný výsledek, můžeš nastavit seed pro pseudonáhodný generátor čísel pomocí argumentu `seed_transpiler`. Opakované spuštění se stejným seedem vrátí stejné výsledky.

Příklad:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Počáteční rozvržení
Před transpilací jsou Qubit obsažené v tvém Circuit virtuální Qubit, které nemusí nutně odpovídat fyzickým Qubit na cílovém Backend. Počáteční mapování virtuálních Qubit na fyzické Qubit můžeš zadat pomocí argumentu `initial_layout`. Vezmi na vědomí, že konečné rozvržení Qubit se může lišit od počátečního rozvržení, protože Transpiler může Qubit permutovat pomocí swap Gate nebo jiných prostředků.

V níže uvedeném příkladu sestavíme počáteční rozvržení pro mock Backend [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) vytvořením objektu [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Naše rozvržení mapuje první Qubit našeho Circuit na Qubit 5 Sherbrooke a druhý Qubit našeho Circuit na Qubit 6 Sherbrooke. Fyzické Qubit jsou vždy reprezentovány celými čísly.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Kromě zadání objektu Layout můžeš také předat seznam celých čísel, kde $i$-tý prvek seznamu obsahuje fyzický Qubit, na který má být mapován $i$-tý Qubit. Například:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Pomocí funkce [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) můžeš vygenerovat diagram grafu zařízení s informacemi o chybách a s označenými fyzickými Qubit. Podobné diagramy si také můžeš prohlédnout na stránce [Compute resources](https://quantum.cloud.ibm.com/computers).